# 6. DiD with covariates: minimum wage and teen employment (staggered adoption)

**Question.** Did state minimum-wage increases reduce teen employment in the counties of those
states? Counties were treated in 2004, 2006 or 2007, or never, between 2003 and 2007.

**Data.** `mpdta`, the county panel of Callaway & Sant'Anna (2021): 500 counties, log teen
employment (`lemp`), log population in 2000 (`lpop`, time-invariant) and the first treated
year (`first.treat`, 0 = never). It ships with the R package `did` (GPL); to keep this MIT
repository clean of it, the example downloads the CSV copy distributed with the MIT-licensed
Python port `csdid` once and caches it.

**Hypothesis, written before estimating.** A higher minimum wage lowers teen employment (a
negative ATT), growing with exposure. Larger counties may have different employment trends
(urban labour markets), so parallel trends are more credible **conditional on population**.

In [1]:
import io
import contextlib
import urllib.request
from pathlib import Path

import numpy as np

from _setup import *  # noqa: F401,F403
import causal as ca
import reporting as rp

URL = "https://raw.githubusercontent.com/d2cml-ai/csdid/main/data/mpdta.csv"
cache = Path(".cache") / "mpdta.csv"          # examples run from the examples/ folder
if not cache.exists():
    cache.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, cache)
d = pd.read_csv(cache)
print(d.shape, "| counties by first treated year:", d.groupby("first.treat")["countyreal"].nunique().to_dict())

(2500, 6) | counties by first treated year: {0: 309, 2004: 20, 2006: 40, 2007: 131}


## Step 1. The unconditional benchmark
Staggered adoption: Callaway-Sant'Anna with never-treated controls, unit bootstrap.

In [2]:
r0 = ca.att_gt(d, "countyreal", "year", "lemp", "first.treat", n_boot=199)


Callaway-Sant'Anna (never controls, 3 cohorts, 199 bootstrap reps; no covariates)
Overall ATT: -0.0400 (SE 0.0121, 95% CI -0.0636 to -0.0163)
Dynamic (event-time) ATT:
       att     se  ci_low  ci_high
e                                 
-3  0.0305 0.0148  0.0016   0.0594
-2 -0.0006 0.0126 -0.0252   0.0241
-1 -0.0245 0.0147 -0.0533   0.0044
 0 -0.0199 0.0119 -0.0432   0.0033
 1 -0.0510 0.0171 -0.0846  -0.0173
 2 -0.1373 0.0362 -0.2083  -0.0662
 3 -0.1008 0.0319 -0.1634  -0.0383


The overall ATT is about −0.040 (−4% teen employment), the number in Callaway & Sant'Anna's
application. The pre-period cell e = −3 is positive and its CI just excludes zero: the
unconditional trends are not perfectly parallel, a reason to condition on covariates.

## Step 2. Do covariate effects differ across groups or years? (Karim & Webb's CCC)
Before adjusting for population, check that its effect on employment is the same in every
cohort and year among **untreated** county-years. The groups are the adoption cohorts (several
counties per cell, unlike `countyreal`), with SEs clustered by county and wild-bootstrap
p-values.

In [3]:
cc = ca.ccc_test(d, "lemp", ["lpop"], group="first.treat", time="year", gvar="first.treat",
                 cluster="countyreal", n_boot=199)


Common causal covariates test (Karim & Webb), 2209 untreated obs in 13 group-period cells, CR1 by countyreal (500 clusters), wild bootstrap (199 draws)
                                                  stat  df     p                      type  p_holm
test                                                                                              
stage 1: slopes equal in every group-period cell 17.78  12 0.285  Wald CR1, wild bootstrap     NaN
Selected: homogeneous (stage 1 not rejected). Covariate effects look common across groups and periods: TWFE or att_gt with covariates are fine on this front (parallel trends still has to be argued; nonrejection is not proof).


Stage 1 does not reject equal slopes across the 13 untreated cohort-year cells, so the
common-covariate assumption is not contradicted: a covariate-adjusted Callaway-Sant'Anna is
appropriate (with group-specific slopes, intersection DiD would be needed instead).

## Step 3. Callaway-Sant'Anna with the covariate in levels, doubly robust
`lpop` is time-invariant: a TWFE regression could not use it at all (unit fixed effects absorb
it). In each group-time 2×2, `att_gt` compares treated and comparison counties of similar
population, through an outcome regression and a propensity score.

In [4]:
r1 = ca.att_gt(d, "countyreal", "year", "lemp", "first.treat", n_boot=199, covariates=["lpop"])


Callaway-Sant'Anna (never controls, 3 cohorts, 199 bootstrap reps; doubly robust, X: lpop (base))
Overall ATT: -0.0418 (SE 0.0118, 95% CI -0.0649 to -0.0186)
Dynamic (event-time) ATT:
       att     se  ci_low  ci_high
e                                 
-3  0.0267 0.0140 -0.0006   0.0541
-2 -0.0036 0.0125 -0.0280   0.0208
-1 -0.0232 0.0147 -0.0520   0.0055
 0 -0.0211 0.0115 -0.0436   0.0015
 1 -0.0530 0.0166 -0.0856  -0.0204
 2 -0.1404 0.0367 -0.2123  -0.0686
 3 -0.1069 0.0315 -0.1686  -0.0452
Independent recomputation of the overall ATT: dr -0.0418, reg -0.0420, ipw -0.0418


The adjusted ATT is about −0.042, slightly larger in magnitude, with the effect growing to
−10% to −14% two to three years after adoption. The pre-period cell at e = −3 shrinks and its
CI now includes zero. The built-in independent recomputation (doubly robust, outcome
regression and IPW) agrees to the third decimal, so the two nuisance models tell the same
story.

## Step 4. Verify against an independent implementation
The same estimator in another code base: the `csdid` port of the R `did`/`DRDID` packages,
when installed. `reporting.cross_check` with no SE demands agreement to numerical precision.

In [5]:
try:
    from csdid.att_gt import ATTgt
    with contextlib.redirect_stdout(io.StringIO()):
        a = ATTgt(yname="lemp", gname="first.treat", idname="countyreal", tname="year",
                  xformla="lemp~lpop", data=d).fit(est_method="dr")
    cs = pd.DataFrame({"g": a.results["group"], "t": a.results["year"], "csdid": a.results["att"]})
    both = r1["gt"].reset_index().merge(cs, on=["g", "t"])
    worst = float(np.max(np.abs(both["att"] - both["csdid"])))
    print(f"Largest cell difference vs csdid: {worst:.2e} over {len(both)} group-time cells")
    rp.cross_check({"this skill (cell 2004/2007)": both.query("g == 2004 and t == 2007")["att"].iloc[0],
                    "csdid (cell 2004/2007)": both.query("g == 2004 and t == 2007")["csdid"].iloc[0]},
                   label="ATT(2004, 2007)")
except ImportError:
    print("csdid not installed: pip install csdid to run the external check")

Largest cell difference vs csdid: 4.74e-15 over 12 group-time cells

Independent check of the ATT(2004, 2007): this skill (cell 2004/2007) -0.106904, csdid (cell 2004/2007) -0.106904 → agree (to a relative tolerance of 1e-06)


## Step 5. What would a TWFE regression with the covariate have said?

In [6]:
tw = ca.twfe_covariate_diagnostic(d, "countyreal", "year", "lemp", "first.treat", ["lpop"], n_boot=199)


TWFE with covariates vs conditional parallel trends (lpop)
TWFE: -0.0365 (SE 0.0148)   att_gt (dr, both): -0.0418 (SE 0.0118)   gap +0.0052 (+0.4 SE)
Hidden linearity test (levels of X in untreated trends): joint p = 0.0713
         coef     se      p
L_lpop 0.0051 0.0028 0.0713
Treated observations with negative TWFE weight: 6.9%
Verdict: TWFE with covariates is NOT reliable here; report att_gt with covariates.


Three red flags: `lpop` drops out of TWFE entirely; with staggered adoption some treated
county-years get **negative** weight (already-treated counties serve as controls); and the
level of population predicts untreated employment trends at the 10% level, the information
TWFE throws away. Here the TWFE number happens to be close (−0.037), but it rests on
assumptions the data do not support; the estimate to report is Callaway-Sant'Anna with the
covariate.

## Step 6. Is the result driven by the exact dates?

In [7]:
ts = ca.did_timing_sensitivity(d, "countyreal", "year", "lemp", "first.treat", max_shift=1, n_boot=99,
                               covariates=["lpop"])


Timing sensitivity (Callaway-Sant'Anna, dates shifted by -1…+1 periods)
          att     se  ci_low  ci_high      p
shift                                       
-1    -0.0356 0.0135 -0.0620  -0.0091 0.0084
0     -0.0418 0.0120 -0.0652  -0.0183 0.0005
1     -0.0729 0.0158 -0.1039  -0.0420 0.0000
Largest swing vs recorded dates: 75% of the baseline ATT (mechanical: a wrong date puts treated periods in the base or untreated ones in the post period, so swings are expected; the date-robust line below is the verdict)
Date-robust ATT (true date anywhere within ±1): -0.0475 (SE 0.0239, 95% CI -0.0944 to -0.0006); cohorts dropped: [2005]
  → the conclusion survives date errors of up to ±1 periods.


Shifting the dates moves the ATT mechanically, as expected. The verdict is the date-robust
estimate, which assumes each true date lies within one year of the recorded one: it stays
negative with a CI that excludes zero, so the conclusion survives date errors of ±1 year.

## Step 7. Report, with a run manifest

*Minimum-wage increases reduced teen employment in treated counties by about 4% on average
over 2004–2007 (doubly robust Callaway-Sant'Anna conditional on county population; never-
treated counties as controls; county bootstrap), with effects growing to 10–14% after two to
three years. Covariate effects are common across cohorts and years (CCC test not rejected).
The estimate matches an independent implementation cell by cell. Limits: parallel trends
conditional on population is an assumption; one pre-period cell is borderline; county-level
spillovers are ignored; the result's sensitivity to the exact dates is shown above.*

In [8]:
man = rp.run_manifest(data={"mpdta": d}, files=[cache], seed=0,
                      results={"ATT unconditional": r0["overall"], "ATT conditional on lpop (DR)": r1["overall"]})


Run manifest (skill 2.3.0, Python 3.13.18, seed 0)
  data mpdta: 2500x6, sha256 b8083b0bc24dde83…
  file .cache/mpdta.csv: sha256 2283bea1221a1524…
  packages: numpy 2.5.3, pandas 3.0.5, scipy 1.18.1, statsmodels 0.15.0, linearmodels 7.1, arch 8.0.0, scikit-learn 1.9.1
